In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table_name = (
    "global_economic_analytics.bronze.world_bank_raw"
)

silver_table_name = (
    "global_economic_analytics.silver.macro_observation"
)

print(bronze_table_name)
print(silver_table_name)

In [0]:
df_bronze = spark.table(bronze_table_name)

print(df_bronze.count())

In [0]:
display(
    df_bronze.select(
        "ingestion_id",
        "ingestion_timestamp",
        "country_code",
        "indicator_code",
        "http_status_code"
    )
)

In [0]:
raw_payload_preview = (
    df_bronze
    .select("raw_payload")
    .first()["raw_payload"][:500]
)

print(raw_payload_preview)

In [0]:
# json_schema = (
#     spark.read
#     .json(
#         spark.sparkContext.parallelize(
#             [raw_payload_preview]
#         )
#     )
#     .schema
# )

# print(json_schema.simpleString())

In [0]:
raw_payload_complete = (
    df_bronze
    .select("raw_payload")
    .first()["raw_payload"]
)

json_schema = spark.range(1).select(
    F.schema_of_json(
        F.lit(raw_payload_complete)
    ).alias("json_schema")
).first()["json_schema"]

print(json_schema)

In [0]:
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType
)

In [0]:
indicator_schema = StructType([
    StructField("id", StringType(), True),
    StructField("value", StringType(), True)
])

country_schema = StructType([
    StructField("id", StringType(), True),
    StructField("value", StringType(), True)
])

In [0]:
observation_schema = StructType([
    StructField("indicator", indicator_schema, True),
    StructField("country", country_schema, True),
    StructField("countryiso3code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("obs_status", StringType(), True),
    StructField("decimal", LongType(), True)
])

In [0]:
observations_schema = ArrayType(
    observation_schema,
    containsNull=True
)

print(observations_schema.simpleString())

In [0]:
df_observations_array = (
    df_bronze
    .withColumn(
        "observations_json",
        F.get_json_object(
            F.col("raw_payload"),
            "$[1]"
        )
    )
    .withColumn(
        "observations",
        F.from_json(
            F.col("observations_json"),
            observations_schema
        )
    )
)

display(
    df_observations_array.select(
        "ingestion_id",
        F.size("observations").alias("observation_count")
    )
)

In [0]:
df_observations_exploded = (
    df_observations_array
    .withColumn(
        "observation",
        F.explode(F.col("observations"))
    )
)

print(df_observations_exploded.count())

In [0]:
display(
    df_observations_exploded.select(
        "ingestion_id",
        "observation"
    ).limit(1)
)

In [0]:
df_silver_base = (
    df_observations_exploded
    .select(
        F.col("ingestion_id"),
        F.col("ingestion_timestamp"),
        F.col("observation.countryiso3code").alias("country_code"),
        F.col("observation.country.value").alias("country_name"),
        F.col("observation.indicator.id").alias("indicator_code"),
        F.col("observation.indicator.value").alias("indicator_name"),
        F.col("observation.date").cast("int").alias("year"),
        F.col("observation.value").alias("indicator_value"),
        F.col("observation.unit").alias("unit"),
        F.col("observation.obs_status").alias("observation_status"),
        F.col("observation.decimal").cast("int").alias("decimal_places")
    )
)

df_silver_base.printSchema()

In [0]:
display(
    df_silver_base
    .select(
        "country_code",
        "country_name",
        "indicator_code",
        "indicator_name",
        "year",
        "indicator_value"
    )
    .orderBy(F.col("year").desc())
    .limit(5)
)

In [0]:
silver_record_count = df_silver_base.count()

print(silver_record_count)

In [0]:
df_silver_duplicates = (
    df_silver_base
    .groupBy(
        "country_code",
        "indicator_code",
        "year"
    )
    .count()
    .filter(F.col("count") > 1)
)

silver_duplicate_count = df_silver_duplicates.count()

print(silver_duplicate_count)

In [0]:
silver_null_key_count = (
    df_silver_base
    .filter(
        F.col("country_code").isNull()
        | F.col("indicator_code").isNull()
        | F.col("year").isNull()
    )
    .count()
)

print(silver_null_key_count)

In [0]:
silver_null_value_count = (
    df_silver_base
    .filter(F.col("indicator_value").isNull())
    .count()
)

print(silver_null_value_count)

In [0]:
year_range = (
    df_silver_base
    .agg(
        F.min("year").alias("minimum_year"),
        F.max("year").alias("maximum_year")
    )
    .first()
)

print(year_range)

In [0]:
distinct_year_count = (
    df_silver_base
    .select("year")
    .distinct()
    .count()
)

expected_year_count = (
    year_range["maximum_year"]
    - year_range["minimum_year"]
    + 1
)

print(distinct_year_count)
print(expected_year_count)

In [0]:
assert (
    distinct_year_count == expected_year_count
), "A série histórica possui anos ausentes"

print("Continuidade anual validada com sucesso!")

In [0]:
df_silver = (
    df_silver_base
    .withColumn(
        "observation_key",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("country_code"),
                F.col("indicator_code"),
                F.col("year").cast("string")
            ),
            256
        )
    )
)

print(df_silver.columns)

In [0]:
observation_key_summary = (
    df_silver
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("observation_key").alias("distinct_keys")
    )
    .first()
)

print(observation_key_summary)

In [0]:
df_silver_organizado = df_silver.select(
    "observation_key",
    "country_code",
    "country_name",
    "indicator_code",
    "indicator_name",
    "year",
    "indicator_value",
    "unit",
    "observation_status",
    "decimal_places",
    "ingestion_id",
    "ingestion_timestamp"
)

print(df_silver_organizado.columns)

In [0]:
silver_table_exists = spark.catalog.tableExists(
    silver_table_name
)

print(silver_table_exists)

In [0]:
# (
#     df_silver_organizado.write
#     .format("delta")
#     .mode("errorifexists")
#     .saveAsTable(silver_table_name)
# )

# print("Tabela Silver criada com sucesso!")

In [0]:
(
    df_silver_organizado.write
    .format("delta")
    .saveAsTable(silver_table_name)
)

print("Tabela Silver criada com sucesso!")

In [0]:
silver_table_exists_after_write = spark.catalog.tableExists(
    silver_table_name
)

print(silver_table_exists_after_write)

In [0]:
silver_table_count = spark.table(
    silver_table_name
).count()

print(silver_table_count)

In [0]:
display(
    spark.table(silver_table_name)
    .select(
        "country_code",
        "indicator_code",
        "year",
        "indicator_value",
        "observation_key"
    )
    .orderBy(F.col("year").desc())
    .limit(5)
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE DETAIL {silver_table_name}
    """).select(
        "format",
        "name",
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {silver_table_name}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
)

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {silver_table_name}
    IS 'Observações macroeconômicas do Banco Mundial limpas, tipadas e estruturadas.'
""")

print("Comentário da tabela Silver adicionado com sucesso!")

In [0]:
silver_table_comment = spark.sql("""
    SELECT comment
    FROM global_economic_analytics.information_schema.tables
    WHERE table_schema = 'silver'
      AND table_name = 'macro_observation'
""").first()["comment"]

print(silver_table_comment)

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN observation_key
    COMMENT 'Chave SHA-256 da combinação entre país, indicador e ano.'
""")

print("Coluna observation_key documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN country_code
    COMMENT 'Código ISO 3 do país da observação.'
""")

print("Coluna country_code documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN country_name
    COMMENT 'Nome do país da observação conforme retornado pelo Banco Mundial.'
""")

print("Coluna country_name documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN indicator_code
    COMMENT 'Código do indicador macroeconômico do Banco Mundial.'
""")

print("Coluna indicator_code documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN indicator_name
    COMMENT 'Nome do indicador macroeconômico conforme retornado pelo Banco Mundial.'
""")

print("Coluna indicator_name documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN year
    COMMENT 'Ano de referência da observação macroeconômica.'
""")

print("Coluna year documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN indicator_value
    COMMENT 'Valor numérico do indicador para o país e ano de referência.'
""")

print("Coluna indicator_value documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN unit
    COMMENT 'Unidade de medida informada pela API para o indicador.'
""")

print("Coluna unit documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN observation_status
    COMMENT 'Status da observação informado pela API do Banco Mundial.'
""")

print("Coluna observation_status documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN decimal_places
    COMMENT 'Quantidade de casas decimais informada pela API para a observação.'
""")

print("Coluna decimal_places documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN ingestion_id
    COMMENT 'Identificador do lote Bronze que originou a observação.'
""")

print("Coluna ingestion_id documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    ALTER COLUMN ingestion_timestamp
    COMMENT 'Data e hora em UTC da ingestão do lote Bronze que originou a observação.'
""")

print("Coluna ingestion_timestamp documentada com sucesso!")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE TABLE {silver_table_name}
    """)
)

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table_name}
    SET TBLPROPERTIES (
        'project' = 'global_economic_analytics',
        'data_source' = 'World Bank API',
        'data_layer' = 'silver',
        'data_granularity' = 'one row per country, indicator and year'
    )
""")

print("Propriedades da tabela Silver adicionadas com sucesso!")

In [0]:
silver_table_properties = spark.sql(f"""
    SHOW TBLPROPERTIES {silver_table_name}
""")

display(
    silver_table_properties.filter(
        """
        key IN (
            'project',
            'data_source',
            'data_layer',
            'data_granularity'
        )
        """
    )
)

In [0]:
silver_source_view = "vw_macro_observation_source"

df_silver_organizado.createOrReplaceTempView(
    silver_source_view
)

print(silver_source_view)

In [0]:
silver_source_count = spark.table(
    silver_source_view
).count()

print(silver_source_count)

In [0]:
spark.sql(f"""
    MERGE INTO {silver_table_name} AS target
    USING {silver_source_view} AS source
        ON target.observation_key = source.observation_key

    WHEN MATCHED THEN UPDATE SET
        target.country_code = source.country_code,
        target.country_name = source.country_name,
        target.indicator_code = source.indicator_code,
        target.indicator_name = source.indicator_name,
        target.year = source.year,
        target.indicator_value = source.indicator_value,
        target.unit = source.unit,
        target.observation_status = source.observation_status,
        target.decimal_places = source.decimal_places,
        target.ingestion_id = source.ingestion_id,
        target.ingestion_timestamp = source.ingestion_timestamp

    WHEN NOT MATCHED THEN INSERT (
        observation_key,
        country_code,
        country_name,
        indicator_code,
        indicator_name,
        year,
        indicator_value,
        unit,
        observation_status,
        decimal_places,
        ingestion_id,
        ingestion_timestamp
    )
    VALUES (
        source.observation_key,
        source.country_code,
        source.country_name,
        source.indicator_code,
        source.indicator_name,
        source.year,
        source.indicator_value,
        source.unit,
        source.observation_status,
        source.decimal_places,
        source.ingestion_id,
        source.ingestion_timestamp
    )
""")

print("MERGE Silver executado com sucesso!")

In [0]:
silver_count_after_merge = spark.table(
    silver_table_name
).count()

print(silver_count_after_merge)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {silver_table_name}
    """)
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
    .orderBy(F.col("version").desc())
    .limit(1)
)

In [0]:
view_removed = spark.catalog.dropTempView(
    silver_source_view
)

print(view_removed)


In [0]:
silver_source_view_exists = spark.catalog.tableExists(
    silver_source_view
)

print(silver_source_view_exists)

In [0]:
silver_final_validation = (
    spark.table(silver_table_name)
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("observation_key").alias("distinct_keys"),
        F.countDistinct("country_code").alias("country_count"),
        F.countDistinct("indicator_code").alias("indicator_count"),
        F.min("year").alias("minimum_year"),
        F.max("year").alias("maximum_year")
    )
    .first()
)

print(silver_final_validation)

In [0]:
assert silver_final_validation["total_records"] == 25
assert silver_final_validation["distinct_keys"] == 25
assert silver_final_validation["country_count"] == 1
assert silver_final_validation["indicator_count"] == 1
assert silver_final_validation["minimum_year"] == 2000
assert silver_final_validation["maximum_year"] == 2024

print("Validação final da tabela Silver concluída com sucesso!")